# Atari RL Playground - 完整教程

这个Notebook包含了从环境安装、框架小实验，到调用正式训练脚本的完整流程。

**注意**: 第1步会安装所有Python依赖，第2步会安装 Atari ROM；如果你已经按 README 配好环境，可以直接从第3步开始。

## 第1步：安装依赖

运行以下单元格安装所有必需的库。

In [ ]:
# 安装依赖
import subprocess
import sys

print("正在安装依赖...")
print("这可能需要 5-10 分钟，请耐心等待...\n")

# 安装 Jupyter（如果还没有安装）
try:
    import jupyter
    print("✓ Jupyter 已安装")
except ImportError:
    print("  安装 Jupyter...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jupyter"])
    print("  ✓ Jupyter 安装完成")

# 安装 PyTorch（如果版本不符合）
try:
    import torch
    major, minor = map(int, torch.__version__.split('.')[:2])
    if major >= 2:
        print(f"✓ PyTorch {torch.__version__} 已安装")
    else:
        print(f"  升级 PyTorch...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch>=2.0.0"])
        print("  ✓ PyTorch 升级完成")
except ImportError:
    print("  安装 PyTorch...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch>=2.0.0"])
    print("  ✓ PyTorch 安装完成")

# 安装其他依赖
print("  安装其他依赖...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gymnasium[atari]>=0.28.0"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ale-py>=0.8.0"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "opencv-python", "pyyaml", "tqdm", "pillow", "imageio", "imageio-ffmpeg"])

print("\n✓ 所有依赖安装完成！")

## 第2步：下载Atari ROM文件（如有需要）

大多数情况下，安装 gymnasium[atari] / ale-py 后已经自带常用ROM；只有在创建环境时遇到缺ROM的报错，才需要运行本步骤。


In [ ]:
# 下载Atari ROM（仅在遇到缺ROM错误时运行）
print("如果你在创建 Atari 环境时遇到 ROM 缺失错误，可以运行本单元格。\n")
print("当前命令等价于在终端执行: python -m ale_py.roms_downloader\n")

try:
    subprocess.check_call([sys.executable, "-m", "ale_py.roms_downloader"])
    print("✓ ROM 文件下载完成！")
except Exception as e:
    print(f"⚠️  ROM 下载失败: {e}")
    print("这通常不影响前面的框架演示，你仍然可以运行所有基于模拟数据的实验。")

## 第3步：导入框架

导入所有必需的模块。

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# 添加项目路径
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# 导入框架
try:
    from algorithms import DQNAgent, PPOAgent, EWCWrapper
    from environments import AtariEnv
    from utils import ReplayBuffer, VideoRecorder
    print("✓ 框架导入成功！")
except ImportError as e:
    print(f"✗ 导入失败: {e}")
    print("请确保在项目根目录运行此 Notebook")
    raise

# 显示环境信息
print(f"\nPyTorch版本: {torch.__version__}")
print(f"GPU可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU设备: {torch.cuda.get_device_name(0)}")
else:
    print("将使用 CPU（较慢但可用）")

## 第4步：快速检查

验证所有导入和基本功能是否正常。

In [ ]:
# 快速检查：验证所有导入
print("=" * 60)
print("快速检查：验证所有导入")
print("=" * 60)

checks = [
    ("DQNAgent", DQNAgent),
    ("PPOAgent", PPOAgent),
    ("EWCWrapper", EWCWrapper),
    ("AtariEnv", AtariEnv),
    ("ReplayBuffer", ReplayBuffer),
    ("VideoRecorder", VideoRecorder),
]

all_ok = True
for name, cls in checks:
    try:
        # 实际检查类是否可用
        if cls is None:
            print(f"✗ {name} 导入失败（为 None）")
            all_ok = False
        elif not isinstance(cls, type):
            print(f"⚠️  {name} 不是一个类")
            all_ok = False
        else:
            print(f"✓ {name} 导入成功")
    except Exception as e:
        print(f"✗ {name} 导入失败: {e}")
        all_ok = False

if all_ok:
    print("\n✓ 所有导入检查完成！")
else:
    print("\n✗ 部分导入检查失败，请检查上面的错误信息")

## 第5步：运行框架演示

这会在模拟数据上演示框架的核心模块（DQN / PPO / EWC / ReplayBuffer），与命令行脚本 scripts/train_single.py 和 scripts/train_continual.py 共享同一套代码。


In [ ]:
# 演示1：创建Agent
print("=" * 60)
print("演示1: 创建不同的Agent")
print("=" * 60)

dqn_agent = DQNAgent(state_dim=4, action_dim=18)
ppo_agent = PPOAgent(state_dim=4, action_dim=18)
ewc_agent = EWCWrapper(DQNAgent(state_dim=4, action_dim=18), ewc_lambda=0.4)

print("✓ DQN Agent 创建成功")
print("✓ PPO Agent 创建成功")
print("✓ EWC Agent 创建成功")

In [ ]:
# 演示2：经验回放
print("\n" + "=" * 60)
print("演示2: 经验回放缓冲")
print("=" * 60)

buffer = ReplayBuffer(capacity=1000)

# 添加随机经验
for _ in range(100):
    state = torch.randn(4, 84, 84)
    action = np.random.randint(18)
    reward = np.random.randn()
    next_state = torch.randn(4, 84, 84)
    done = False
    buffer.add(state, action, reward, next_state, done)

print(f"✓ 添加了100条经验")
print(f"✓ 缓冲区大小: {len(buffer)}")

# 采样批次
batch = buffer.sample(32)
print(f"✓ 采样了32条经验")
print(f"  状态形状: {batch[0].shape}")
print(f"  动作形状: {batch[1].shape}")

In [ ]:
# 演示3：Agent更新
print("\n" + "=" * 60)
print("演示3: Agent学习")
print("=" * 60)

# DQN更新
dqn_metrics = dqn_agent.update(batch)
print(f"DQN更新:")
print(f"  Loss: {dqn_metrics['loss']:.4f}")
print(f"  Epsilon: {dqn_metrics['epsilon']:.4f}")

# PPO更新
ppo_metrics = ppo_agent.update(batch)
print(f"\nPPO更新:")
print(f"  Policy Loss: {ppo_metrics['policy_loss']:.4f}")
print(f"  Value Loss: {ppo_metrics['value_loss']:.4f}")
print(f"  Entropy: {ppo_metrics['entropy']:.4f}")

In [ ]:
# 演示4：灾难性遗忘
print("\n" + "=" * 60)
print("演示4: 灾难性遗忘(无EWC)")
print("=" * 60)

agent = DQNAgent(state_dim=4, action_dim=18)
buffer = ReplayBuffer(capacity=5000)

# Task 1
print("\nTask 1: 学习任务1")
losses_task1 = []
for step in range(500):
    state = torch.randn(4, 84, 84)
    action = np.random.randint(18)
    reward = np.random.randn()
    next_state = torch.randn(4, 84, 84)
    done = False
    buffer.add(state, action, reward, next_state, done)
    
    if buffer.is_ready(32):
        batch = buffer.sample(32)
        metrics = agent.update(batch)
        losses_task1.append(metrics['loss'])

avg_loss_task1 = np.mean(losses_task1[-100:])
print(f"Task 1 最终Loss: {avg_loss_task1:.4f}")

# Task 2
print("\nTask 2: 学习任务2(不同分布)")
losses_task2 = []
for step in range(500):
    state = torch.randn(4, 84, 84) * 2  # 不同分布
    action = np.random.randint(18)
    reward = np.random.randn() * 2
    next_state = torch.randn(4, 84, 84) * 2
    done = False
    buffer.add(state, action, reward, next_state, done)
    
    if buffer.is_ready(32):
        batch = buffer.sample(32)
        metrics = agent.update(batch)
        losses_task2.append(metrics['loss'])

avg_loss_task2 = np.mean(losses_task2[-100:])
print(f"Task 2 最终Loss: {avg_loss_task2:.4f}")

print(f"\n⚠️  Loss从 {avg_loss_task1:.4f} 增加到 {avg_loss_task2:.4f}")
print("这表明Agent在学习Task 2时忘记了Task 1")

In [ ]:
# 演示5：EWC缓解遗忘
print("\n" + "=" * 60)
print("演示5: EWC缓解灾难性遗忘")
print("=" * 60)

base_agent = DQNAgent(state_dim=4, action_dim=18)
ewc_agent = EWCWrapper(base_agent, ewc_lambda=0.4)
buffer = ReplayBuffer(capacity=5000)

# Task 1
print("\nTask 1: 学习任务1")
losses_task1_ewc = []
for step in range(500):
    state = torch.randn(4, 84, 84)
    action = np.random.randint(18)
    reward = np.random.randn()
    next_state = torch.randn(4, 84, 84)
    done = False
    buffer.add(state, action, reward, next_state, done)
    
    if buffer.is_ready(32):
        batch = buffer.sample(32)
        metrics = ewc_agent.update(batch)
        losses_task1_ewc.append(metrics['loss'])

avg_loss_task1_ewc = np.mean(losses_task1_ewc[-100:])
print(f"Task 1 最终Loss: {avg_loss_task1_ewc:.4f}")

# 巩固权重
print("\n正在巩固权重(计算Fisher信息矩阵)...")
ewc_agent.consolidate_weights()
print("✓ 权重巩固完成")

# Task 2
print("\nTask 2: 学习任务2(不同分布)")
losses_task2_ewc = []
ewc_losses = []
for step in range(500):
    state = torch.randn(4, 84, 84) * 2
    action = np.random.randint(18)
    reward = np.random.randn() * 2
    next_state = torch.randn(4, 84, 84) * 2
    done = False
    buffer.add(state, action, reward, next_state, done)
    
    if buffer.is_ready(32):
        batch = buffer.sample(32)
        metrics = ewc_agent.update(batch)
        losses_task2_ewc.append(metrics['loss'])
        if 'ewc_loss' in metrics:
            ewc_losses.append(metrics['ewc_loss'])

avg_loss_task2_ewc = np.mean(losses_task2_ewc[-100:])
avg_ewc_loss = np.mean(ewc_losses) if ewc_losses else 0
print(f"Task 2 最终Loss: {avg_loss_task2_ewc:.4f}")
print(f"平均EWC正则化Loss: {avg_ewc_loss:.6f}")

print(f"\n✓ EWC正则化项保护了Task 1的重要权重")
print(f"  Loss从 {avg_loss_task1_ewc:.4f} 变为 {avg_loss_task2_ewc:.4f}")
print(f"  相比无EWC的 {avg_loss_task2:.4f}，改善明显")

In [ ]:
# 绘制对比图
print("\n" + "=" * 60)
print("演示6: 可视化对比")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 无EWC
axes[0].plot(losses_task1, label='Task 1', alpha=0.7)
axes[0].plot(range(len(losses_task1), len(losses_task1) + len(losses_task2)), 
             losses_task2, label='Task 2', alpha=0.7)
axes[0].axvline(x=len(losses_task1), color='red', linestyle='--', label='Task切换')
axes[0].set_title('无EWC - 灾难性遗忘')
axes[0].set_xlabel('步数')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# 有EWC
axes[1].plot(losses_task1_ewc, label='Task 1', alpha=0.7)
axes[1].plot(range(len(losses_task1_ewc), len(losses_task1_ewc) + len(losses_task2_ewc)), 
             losses_task2_ewc, label='Task 2', alpha=0.7)
axes[1].axvline(x=len(losses_task1_ewc), color='red', linestyle='--', label='Task切换')
axes[1].set_title('有EWC - 缓解遗忘')
axes[1].set_xlabel('步数')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('ewc_comparison.png', dpi=100)
plt.show()

print("✓ 对比图已保存为 ewc_comparison.png")

## 第6步：训练真实游戏（推荐使用命令行脚本）

在实际教学/实验中，推荐使用命令行脚本来训练真实的 Atari 游戏：

- 单任务训练（DQN / PPO）：`python scripts/train_single.py --game Pong-v5 --algorithm dqn`  等
- 连续多任务训练（含灾难性遗忘 / EWC 展示）：`python scripts/train_continual.py --use-ewc`

下面保留一个简单的 Notebook 内联示例，方便你快速在小步数上做演示。
**注意**: 这会花费一定时间（取决于你的硬件），并且不包含完整的视频/曲线保存逻辑，完整版本请参考上面的命令行脚本。

In [ ]:
# 可选：训练单个游戏
# 取消注释以下代码来训练

# print("开始训练DQN在Pong上...")
# env = AtariEnv("Pong-v5")
# agent = DQNAgent(state_dim=4, action_dim=env.action_space)
# buffer = ReplayBuffer(capacity=100000)
# 
# state = env.reset()
# episode_rewards = []
# current_reward = 0
# 
# for step in tqdm(range(10000)):  # 短训练用于演示
#     action = agent.select_action(state)
#     next_state, reward, done, _ = env.step(action)
#     current_reward += reward
#     
#     buffer.add(state, action, reward, next_state, done)
#     
#     if buffer.is_ready(32):
#         batch = buffer.sample(32)
#         agent.update(batch)
#     
#     state = next_state
#     
#     if done:
#         episode_rewards.append(current_reward)
#         current_reward = 0
#         state = env.reset()
# 
# env.close()
# print(f"✓ 训练完成！平均奖励: {np.mean(episode_rewards):.2f}")